# Veritas executor: end-to-end test

Automated evaluation of lab genome submissions (PathoEQA + Veritas)

## What are we building?

- PathoEQA receives genome submissions from laboratories
- Veritas scores each submission against a known-truth reference
- The **executor** is the piece in between: it takes a work order, runs Veritas on a cloud machine, and reports back
- Goal: no heavy analysis on the web server, and every step traceable

## What this test shows

- A real, public sample (SARS-CoV-2, Gamma variant); the lab submission is **simulated** with known errors
- The PathoEQA server is replaced by a stand-in, so the executor can be tested on its own
- The executor: receives the work order, downloads and verifies every input, runs Veritas, reports progress step by step
- A corrupted input is caught **before** any analysis runs

## Live: start the stand-in server

In [2]:
import subprocess, sys, os, time
MOCK = os.path.expanduser("~/Workspace/veritas/pathoeqa-mock")
server = subprocess.Popen([sys.executable, "mock_pathoeqa_server.py"], cwd=MOCK,
                          stdout=open("/tmp/mock.log", "w"), stderr=subprocess.STDOUT)
time.sleep(1.5)
print(open("/tmp/mock.log").read())

Mock PathoEQA on https://localhost:8443
  good attempt: 11111111-1111-4111-8111-111111111111
  bad  attempt: 22222222-2222-4222-8222-222222222222



## Live: run one evaluation

In [4]:
%%bash
cd ~/Workspace/veritas/pathoeqa-mock && ./run_demo.sh good

2026-10-06 11:35:01,844 INFO veritas_runner: Using OIDC token supplied by the caller.
2026-10-06 11:35:01,845 INFO veritas_runner.pathoeqa: Fetching manifest for attempt_id=11111111-1111-4111-8111-111111111111
2026-10-06 11:35:01,853 INFO veritas_runner.pathoeqa: Callback event_type=attempt_started attempt_id=11111111-1111-4111-8111-111111111111 sample_run_id=None event_id=a33a8480-ebdb-41e4-b479-fc4eb78288d0
2026-10-06 11:35:01,858 INFO veritas_runner.runner: Storage check passed for '/tmp/veritas-demo/11111111-1111-4111-8111-111111111111/out': 341.00 GiB free (required estimate: 50.00 GiB)
2026-10-06 11:35:01,858 INFO veritas_runner.pathoeqa: Callback event_type=sample_started attempt_id=11111111-1111-4111-8111-111111111111 sample_run_id=SEARCH-8113 event_id=9198c2cb-ed7a-499d-b31f-5d1f098d5306
2026-10-06 11:35:01,863 INFO veritas_runner.artefacts: Skipping download role=truth_vcf; cached file verified at /tmp/veritas-demo/11111111-1111-4111-8111-111111111111/out/SEARCH-8113/truth_vc

exit code: 0   result: /tmp/veritas-demo-good.json
{"attempt_id":"11111111-1111-4111-8111-111111111111","terminal_state":"attempt_completed","failure_class":"success","duration_ms":1943,"veritas_version":"unknown","samples_total":1,"samples_completed":1,"dry_run":false}


In [6]:
%%bash
# What the server received, in order
grep CALLBACK /tmp/mock.log | tac | awk '{print} /attempt_started/{exit}' | tac

<<< CALLBACK #5 attempt_started  sample=None  idem=a33a8480-ebdb-41e4-b479-fc4eb78288d0
<<< CALLBACK #6 sample_started  sample=SEARCH-8113  idem=9198c2cb-ed7a-499d-b31f-5d1f098d5306
<<< CALLBACK #7 sample_completed  sample=SEARCH-8113  idem=86e2e167-ee3a-444b-8868-f2a48ee59bea
<<< CALLBACK #8 attempt_completed  sample=None  idem=22e868cf-4541-403c-b93d-08941e1540b5


In [7]:
%%bash
# Variant-calling metrics produced by Veritas
column -t -s$'\t' /tmp/veritas-demo/11111111-1111-4111-8111-111111111111/out/SEARCH-8113/output/metrics.tsv

Category  Region      TP  FP  FN  Precision           Recall              F1-Score
SNV       ALL         32  1   2   0.9696969696969697  0.9411764705882353  0.955223880597015
INDEL     ALL         2   0   0   1.0                 1.0                 1.0
SNV       PRIMER BED  8   0   1                                           
INDEL     PRIMER BED  1   0   0                                           


## Live: a corrupted input (optional)

`checksum_mismatch`: replaced the truth VCF's real SHA-256 with 64 zeros in the manifest, simulating a file that was corrupted in transit or swapped for another one

In [ ]:
%%bash
cd ~/Workspace/veritas/pathoeqa-mock && ./run_demo.sh bad 2>&1 | tail -4
echo; grep CALLBACK /tmp/mock.log | tail -3

## What we gained

- The executor works end to end **before** the real server exists: less integration risk later
- Every input is checked against a fingerprint; bad data stops the run safely
- Progress is reported per step, so partial work is not lost
- Clear, traceable results per sample

## Next steps

1. Bring the PathoEQA server online (deployment in progress)
2. Adopt the finalized data contract between PathoEQA and Veritas (SPEC-16)
3. Harden the code: review, automated tests, cleaner error reporting
4. Run the full chain on GitHub Actions together with PathoEQA
5. Send results back to PathoEQA and show them to administrators
6. Use real laboratory data only after the formal confidentiality decision

In [ ]:
server.terminate()  # stop the stand-in server